# 02b — 48h-ahead Training (light)

Same features and split as the 24h pipeline, target shifted **48 hours** ahead.

**Prerequisite:** `01_preprocessing.ipynb` (uses `forecast_ready.parquet`).

This notebook trains a smaller set of models (Persistence, Ridge, RF, LightGBM) for speed.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = Path("processed_data/forecast_ready.parquet")
META_PATH = Path("processed_data/preprocessing_meta.json")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_parquet(DATA_PATH)
with open(META_PATH, encoding="utf-8") as f:
    meta = json.load(f)

features = meta["features_24h"]  # same feature set

# Build 48h target from raw power if needed
if "target_48h" not in df.columns:
    # P_dusty at t is known; we need power at t+48 as label for row t
    # In preprocessing, target_24h = P.shift(-24). Recompute 48h the same way from residual series.
    if "P_dusty_W_lag_1h" in df.columns:
        # recover approximate current power from lag structure is fragile; use target_24h shift
        df["target_48h"] = df["target_24h"].shift(-24)
    else:
        raise ValueError("Cannot derive target_48h")

df48 = df.dropna(subset=["target_48h"]).copy()

train_df = df48[df48["split"] == "train"]
val_df   = df48[df48["split"] == "val"]
test_df  = df48[df48["split"] == "test"]

X_train, y_train = train_df[features], train_df["target_48h"]
X_val,   y_val   = val_df[features],   val_df["target_48h"]
X_test,  y_test  = test_df[features],  test_df["target_48h"]

day_val  = val_df["is_daytime"].values.astype(bool)
day_test = test_df["is_daytime"].values.astype(bool)

print("48h shapes Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)

48h shapes Train/Val/Test: (2990, 47) (640, 47) (618, 47)


In [2]:
def metrics(y_true, y_pred, name, day_mask):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    yt, yp = y_true[day_mask], y_pred[day_mask]
    ok = yt > 0.5
    mape = np.mean(np.abs((yt[ok] - yp[ok]) / yt[ok])) * 100 if ok.any() else np.nan
    return {"model": name, "MAE": mae, "RMSE": rmse, "MAPE_day": mape, "R2": r2}

def show(m):
    print(f"{m['model']:28s} MAE={m['MAE']:.3f} RMSE={m['RMSE']:.3f} MAPE_day={m['MAPE_day']:.1f}% R2={m['R2']:.4f}")

In [3]:
# Persistence: power 48h ago as prediction of power 48h ahead ≈ lag 48 of current series
persist_col = "P_dusty_W_lag_48h" if "P_dusty_W_lag_48h" in X_val.columns else "P_dusty_W_lag_24h"
m_p_val = metrics(y_val, X_val[persist_col], "Persistence-48h (val)", day_val)
m_p_test = metrics(y_test, X_test[persist_col], "Persistence-48h (test)", day_test)
show(m_p_val); show(m_p_test)

Persistence-48h (val)        MAE=1.088 RMSE=4.201 MAPE_day=13.6% R2=0.9539
Persistence-48h (test)       MAE=1.104 RMSE=3.670 MAPE_day=11.9% R2=0.9718


In [4]:
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE))
ridge.fit(X_train, y_train)
m_ridge = metrics(y_val, ridge.predict(X_val), "Ridge (val)", day_val)
show(m_ridge)

rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
m_rf = metrics(y_val, rf.predict(X_val), "RF (val)", day_val)
show(m_rf)

lgbm = lgb.LGBMRegressor(
    n_estimators=400, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
)
lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(30, verbose=False)])
m_lgbm = metrics(y_val, lgbm.predict(X_val), "LightGBM (val)", day_val)
show(m_lgbm)

Ridge (val)                  MAE=3.207 RMSE=4.382 MAPE_day=51.8% R2=0.9499
RF (val)                     MAE=1.363 RMSE=3.666 MAPE_day=15.7% R2=0.9649
LightGBM (val)               MAE=1.350 RMSE=3.590 MAPE_day=15.4% R2=0.9664


In [5]:
board = pd.DataFrame([m_p_val, m_ridge, m_rf, m_lgbm]).sort_values("MAE")
print(board.to_string(index=False))
board.to_csv("model_comparison_val_48h.csv", index=False)

learned = board[~board["model"].str.contains("Persistence")]
best_name = learned.iloc[0]["model"]
final = {"Ridge (val)": ridge, "RF (val)": rf, "LightGBM (val)": lgbm}[best_name]
print("Selected:", best_name)

pred_test = final.predict(X_test)
m_test = metrics(y_test, pred_test, f"{best_name} → TEST", day_test)
show(m_test); show(m_p_test)

joblib.dump(final, MODEL_DIR / "best_forecast_48h.joblib")
joblib.dump(features, MODEL_DIR / "best_forecast_48h_features.joblib")
print("Saved models/best_forecast_48h.joblib")

                model      MAE     RMSE  MAPE_day       R2
Persistence-48h (val) 1.088359 4.200871 13.623886 0.953922
       LightGBM (val) 1.349764 3.589778 15.433064 0.966352
             RF (val) 1.362891 3.665740 15.704887 0.964913
          Ridge (val) 3.207346 4.381773 51.769068 0.949868
Selected: LightGBM (val)
LightGBM (val) → TEST        MAE=1.907 RMSE=4.467 MAPE_day=15.5% R2=0.9582
Persistence-48h (test)       MAE=1.104 RMSE=3.670 MAPE_day=11.9% R2=0.9718
Saved models/best_forecast_48h.joblib
